In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
# os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")

In [8]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)

In [9]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Ahmed i am a cheif ai engineer")])

AIMessage(content="Nice to meet you, Ahmed. As a Chief AI Engineer, you must be working on some exciting projects. Can you tell me a bit about what you're currently working on or any interesting challenges you're facing in the field of AI?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 48, 'total_tokens': 97, 'completion_time': 0.085226841, 'completion_tokens_details': None, 'prompt_time': 0.00425553, 'prompt_tokens_details': None, 'queue_time': 0.04913171, 'total_time': 0.089482371}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fa8e1-22e8-7c33-8649-953499d6fe6a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 49, 'total_tokens': 97})

### basic single session wrapper-> [] memeory!!!

In [25]:
# from langchain_core.messages import AIMessage
# #list of messages
# model.invoke(
#     [
#         HumanMessage(content="Hi, My name is Ahmed i am a cheif ai engineer"),
#         AIMessage(content="Nice to meet you, Ahmed. As a Chief AI Engineer, you must be working on some exciting projects. Can you tell me a bit about what you're currently working on or any interesting challenges you're facing in the field of AI?.\n"),
#         HumanMessage(content="Hey What's my nmae and what do i do?")
#     ]
# )

### chat_message_history , basicchatmessagehistory, runnablewithmessagehistory

In [26]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store={}
#func to check if its in the history then add if not there!
def get_session_history(session_id:str)->BaseChatMessageHistory: #for storing chat message history
    if session_id not in store:
        store[session_id]=ChatMessageHistory() #store message in memory also rertrive
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

# model — your LLM/chat model (e.g. ChatOpenAI(), ChatGroq(), etc.) — the actual "brain" that generates responses.
# get_session_history — the function you wrote, passed in by reference (not called). RunnableWithMessageHistory will call it internally whenever it needs a session's history.
# with_message_history — a new runnable that behaves just like model, but now automatically wraps every call with history: fetch past messages → send to model → save the new exchange back.

In [27]:
# ChatMessageHistory() — this is a simple object that holds a list of messages
#  (HumanMessage, AIMessage, etc.) 
# for one conversation, and exposes methods like .add_user_message(), 
# .add_ai_message(), and .messages to retrieve them all.

In [28]:
#hard coding tracking id 
config={"configurable":{"session_id":"chat1"}}

In [31]:
resp=with_message_history.invoke(
    [HumanMessage(content="Hi, my name ahmed and i am a chief ai engineer")],
    config=config
)

resp.content

"Nice to meet you, Ahmed. As a Chief AI Engineer, you must be involved in some exciting and cutting-edge projects. What specific aspects of AI interest you the most, or are you working on any projects that you'd like to share?"

In [34]:
#change session_id(config)
config3={"configurable":{"session_id":"chat3"}}
resp3=with_message_history.invoke(
    [HumanMessage(content="what is my name adn what i do?")],
    config=config3
)

resp3.content

"I don't actually know your name or what you do. I'm a conversational AI, and our conversation has just started. I don't have the ability to access any external information about you, and I don't retain information from previous conversations.\n\nIf you'd like, I can suggest some fun options:\n\n1. You can share your name and what you do, and I can chat with you about it.\n2. I can give you a fictional name and occupation, and we can have a fun conversation about it.\n3. If you want, I can ask you a series of questions to try to guess your name and occupation (this would be just for fun, and I wouldn't actually know the correct answers).\n\nLet me know which option sounds interesting to you!"

In [ ]:
resp4=with_message_history.invoke(
    [HumanMessage(content="Whats my name and what i do?")],
    config=config
)

resp4.content #it remebers from config history!

'Your name is Ahmed, and you are a Chief AI Engineer.'